# Lab 21 — Fine-tuning LLMs · RUN ALL + BONUS B1, B4, B5 (T4)

Chạy từ trên xuống. **Runtime > Change runtime type > T4 GPU** trước khi bắt đầu.

| Ô | Làm gì | Thời gian | Ghi chú |
|---|---|---|---|
| 1 | Setup: clone + install | ~1 phút | Cài dependencies tương thích |
| 2 | Smoke test | ~30 giây | Import + Data + Unit tests |
| 3 | **Core Pipeline (NB1 -> NB5) + NB6 (Bonus B1)** | ~90 phút | Core + Merge & Hot-swap adapter |
| 4 | **Bonus B4: Quét Rank có kiểm soát (r=8, 64)** | ~25 phút | Đo lường mức độ đòn bẩy của Rank |
| 5 | **Bonus B5: Push Adapter lên Hugging Face Hub** | ~1 phút | Đẩy adapter công khai (+2đ) |
| 6 | Gatekeeper Verify + Nén file nộp bài | ~10 giây | Kiểm tra liêm chính và sinh file ZIP |


In [ ]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

# Bạn có thể thay bằng repo fork của bạn nếu có
REPO = "https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/Day21-Track3-Finetuning-Lab")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (không cần GPU)
!python scripts/verify.py --smoke


In [ ]:
# @title 3. Core Pipeline (NB1 -> NB5) + NB6 (Bonus B1: Merge & Hot-swap)
# Lưu ý khi NỘP BÀI CHÍNH THỨC: để EVAL_LIMIT rỗng ("") để chạy đủ 50 mẫu đánh giá.
# Để test nhanh kiểm tra pipeline có thể chọn EVAL_LIMIT = "8".
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = ""          # @param ["", "4", "8", "16", "25"]
STAGES       = "all"       # @param ["all", "nb1 nb2 nb3 nb4 nb5", "nb1", "nb2", "nb3", "nb4", "nb5", "nb6"]

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 4. Bonus B4 — Quét Rank có kiểm soát (r ∈ {8, 64})
!python scripts/run_rank_sweep.py


In [ ]:
# @title 5. Bonus B5 — Push Adapter lên Hugging Face Hub (+2 điểm)
HF_TOKEN = ""         # @param {type:"string"}
HF_REPO_ID = ""       # @param {type:"string"} - Ví dụ: username/lab21-qwen35-triage-vi

if HF_REPO_ID.strip():
    token_flag = f"--token {HF_TOKEN}" if HF_TOKEN.strip() else ""
    !python scripts/push_to_hub.py --repo-id {HF_REPO_ID} {token_flag}
else:
    print("Bỏ qua B5 vì chưa điền HF_REPO_ID. Điền HF_REPO_ID và HF_TOKEN nếu muốn nộp bonus B5.")


In [ ]:
# @title 6. Gatekeeper Verify + Đóng gói file nộp bài
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/

# Nén kết quả lại để bạn tải về máy cập nhật vào local repo
!zip -r /content/results_and_adapters.zip results/ adapters/correct/ LINKS.md 2>/dev/null || zip -r /content/results_and_adapters.zip results/ adapters/correct/
print("\nĐã tạo gói: /content/results_and_adapters.zip — Tải file này về máy để cập nhật vào thư mục kết quả!")
